# 02 - Board clustering (emergent meta archetypes)
Run the cells top to bottom. First time only, install packages in the terminal:
`python -m pip install scikit-learn umap-learn matplotlib seaborn ipykernel`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# finds your data folder automatically
DATA = next(p for p in [Path("data"), Path("../data"), Path("D:/VS Code/data")] if (p / "boards.csv").exists())
boards = pd.read_csv(DATA / "boards.csv")
units = pd.read_csv(DATA / "units.csv")

units["unit"] = (units["unit"].str.replace(r"DA_(18_)?", "", regex=True)
                              .str.replace(r"18(?=[_:|]|$)", "", regex=True))
boards["traits"] = (boards["traits"].str.replace(r"DA_(18_)?", "", regex=True)
                                    .str.replace(r"18(?=[_:|]|$)", "", regex=True))

print(boards.shape, units.shape)
boards.head()

## 1. Sanity checks

In [ ]:
print("Boards per match (should be 8):")
print(boards.groupby("match_id").size().value_counts(), "\n")
print("Set number:", boards["set_number"].unique())
print("\nGame versions:")
print(boards["game_version"].value_counts().head(), "\n")
print("Placement counts (should be ~equal):")
print(boards["placement"].value_counts().sort_index())

## 2. Keep a single patch (controls for balance changes)

In [ ]:
main_version = boards["game_version"].value_counts().idxmax()
print("Using:", main_version)
boards = boards[boards["game_version"] == main_version].copy()
units = units[units["board_id"].isin(boards["board_id"])].copy()
print("Boards kept:", len(boards))

## 3. Feature matrix: one row per board, one column per unit (1 = on the board)
Rare units (under 2% of boards) are dropped so they don't add noise.

In [ ]:
X = units.groupby(["board_id", "unit"])["tier"].max().unstack(fill_value=0)
pres = (X > 0).astype(int)
pres = pres.loc[:, pres.mean() >= 0.02]

boards = boards.set_index("board_id")
common = boards.index.intersection(pres.index)
boards, pres = boards.loc[common], pres.loc[common]
print("Matrix:", pres.shape)

## 4. UMAP + HDBSCAN

In [ ]:
import umap
from sklearn.cluster import HDBSCAN   # built into scikit-learn, no extra install

# 5-D embedding for clustering, 2-D for plotting
emb5 = umap.UMAP(n_components=5, n_neighbors=30, min_dist=0.0,
                 metric="cosine", random_state=42).fit_transform(pres.values)
emb2 = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                 metric="cosine", random_state=42).fit_transform(pres.values)

labels = HDBSCAN(min_cluster_size=200, min_samples=20).fit_predict(emb5)
boards["cluster"] = labels
print(pd.Series(labels).value_counts().sort_index())
print("Noise share:", round((labels == -1).mean(), 3))

In [ ]:
rows = []
for mcs in [100, 150, 200, 300]:
    for ms in [10, 20, 40]:
        lab = HDBSCAN(min_cluster_size=mcs, min_samples=ms).fit_predict(emb5)
        mask = lab != -1
        n = len(set(lab)) - (1 if -1 in lab else 0)
        rows.append(dict(min_cluster_size=mcs, min_samples=ms, clusters=n,
                         noise=round((~mask).mean(), 2),
                         largest=round(pd.Series(lab[mask]).value_counts(normalize=True).max(), 2) if n else np.nan))
pd.DataFrame(rows)

## 5. What is each cluster? (best placing first)

In [ ]:
overall = pres.mean()
rows = []
for c in sorted(set(labels) - {-1}):
    idx = boards.index[boards["cluster"] == c]
    freq = pres.loc[idx].mean()
    lift = (freq - overall).sort_values(ascending=False)
    top = ", ".join(f"{u} ({freq[u]:.0%})" for u in lift.index[:6])
    place = boards.loc[idx, "placement"]
    rows.append(dict(cluster=c, boards=len(idx),
                     avg_place=round(place.mean(), 2),
                     top4_rate=round((place <= 4).mean(), 2),
                     signature_units=top))
profile = pd.DataFrame(rows).sort_values("avg_place")
pd.set_option("display.max_colwidth", 200)
profile

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
noise = labels == -1
ax[0].scatter(emb2[noise, 0], emb2[noise, 1], s=3, c="lightgrey")
sc = ax[0].scatter(emb2[~noise, 0], emb2[~noise, 1], s=4, c=labels[~noise], cmap="tab20")
ax[0].set_title("UMAP of boards, coloured by cluster (grey = noise)")

sns.barplot(data=profile, x="cluster", y="avg_place", order=profile["cluster"], ax=ax[1], color="steelblue")
ax[1].axhline(4.5, color="red", ls="--", label="average (4.5)")
ax[1].set_title("Average placement per cluster (lower = better)")
ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# save for the next notebook (XGBoost + SHAP)
boards.to_csv(DATA / "boards_clustered.csv")
pres.to_csv(DATA / "unit_presence_matrix.csv")

In [ ]:
b = pd.read_csv(DATA / "boards_clustered.csv")
b["when"] = pd.to_datetime(b["game_datetime"], unit="ms")
print("Distinct versions:", b["game_version"].nunique())
print("Date range:", b["when"].min(), "->", b["when"].max())
print("Share of zeros:\n", (b[["players_eliminated", "total_damage_to_players"]] == 0).mean())
print("Rows with augments:", b["augments"].notna().mean())

In [ ]:
extra = (boards[boards.cluster != -1].groupby("cluster")
         .agg(avg_level=("level", "mean"), avg_last_round=("last_round", "mean"),
              avg_units=("n_units", "mean"), avg_gold_left=("gold_left", "mean"))
         .round(2).reset_index())

tr = boards["traits"].fillna("").str.split("|").explode()
tr = tr[tr != ""].str.split(":").str[0]
trait_pres = pd.crosstab(tr.index, tr).clip(upper=1).reindex(boards.index, fill_value=0)

trait_rows = []
for c in sorted(set(boards.cluster) - {-1}):
    idx = boards.index[boards.cluster == c]
    f = trait_pres.loc[idx].mean()
    lift = (f - trait_pres.mean()).sort_values(ascending=False)
    trait_rows.append(dict(cluster=c,
        signature_traits=", ".join(f"{t} ({f[t]:.0%})" for t in lift.index[:4])))

profile2 = (profile.drop(columns="signature_units")
            .merge(extra, on="cluster").merge(pd.DataFrame(trait_rows), on="cluster"))
profile2

In [ ]:
def compare(a, b, n=6):
    fa = pres.loc[boards.index[boards.cluster == a]].mean()
    fb = pres.loc[boards.index[boards.cluster == b]].mean()
    d = (fa - fb).sort_values()
    print(f"More common in cluster {b}:\n{d.head(n).round(2)}")
    print(f"\nMore common in cluster {a}:\n{d.tail(n).round(2)}")

compare(1, 2)
compare(11, 10)
compare(13, 12)

In [ ]:
compare(12, 13)
compare(5, 4)

In [ ]:
u = units.groupby("board_id").agg(
    total_items=("n_items", "sum"),
    n_2star=("tier", lambda t: (t == 2).sum()),
    n_3star=("tier", lambda t: (t >= 3).sum()))
quality = (boards[["cluster", "placement"]].join(u)
           .groupby("cluster")[["total_items", "n_2star", "n_3star", "placement"]]
           .mean().round(2).sort_values("placement"))
quality

In [ ]:
b = boards[boards.cluster != -1].copy()
b["family"] = b["cluster"].replace({13: 12, 4: 5})   # merge each pair into one comp family
same = b.groupby(["match_id", "family"]).size().rename("n_same")
b = b.join(same, on=["match_id", "family"])
b["others_same_comp"] = b["n_same"] - 1

print(b.groupby("others_same_comp")["placement"].agg(["count", "mean"]).round(2))
print()
print(b.groupby("cluster")["others_same_comp"].mean().round(2).sort_values())

In [ ]:
names = {
    12: "Aphelios-KogMaw (flexed)",
    13: "Aphelios-KogMaw (pure)",
    5:  "Ahri-Sett-Morgana (Krug/Ashe)",
    4:  "Ahri-Sett-Morgana (Karma)",
    3:  "KhaZix-Hecarim reroll",
    0:  "RekSai-LeBlanc reroll",
    2:  "Rammus-Teemo reroll",
    6:  "MasterYi-Yorick 3-star",
    11: "Ezreal-Gnar-Alistar",
    7:  "Yorick-Maokai-Zyra",
    1:  "Malphite-Azir-Soraka",
    9:  "Brambleback-CrimsonRaptor hybrid",
    8:  "Scuttlecrab-Yorick-Vi",
    10: "Flex / mixed",
}
boards["archetype"] = boards["cluster"].map(names).fillna("Flex / mixed")
print(boards.groupby("archetype")["placement"].agg(["count", "mean"]).round(2).sort_values("mean"))

boards.to_csv(DATA / "boards_clustered.csv")   # make sure this CSV is closed in Excel first

In [ ]:
b2 = b[b.cluster != 10].copy()
b2["place_vs_cluster"] = b2["placement"] - b2.groupby("cluster")["placement"].transform("mean")
print(b2.groupby("others_same_comp")[["placement", "place_vs_cluster"]]
        .agg(["count", "mean"]).round(2))

In [ ]:
bm = boards.groupby("match_id").size()
print(bm.value_counts())
short = bm[bm < 8].index
for m in short:
    got = set(boards.loc[boards.match_id == m, "placement"])
    print(m, "missing placements:", sorted(set(range(1, 9)) - got))

## Tuning tips
- Too many tiny clusters or too much noise: lower `min_samples`, or lower `min_cluster_size`.
- Everything merges into 1-2 clusters: raise `n_neighbors` slightly or lower `min_cluster_size`.
- Try clustering only top-4 boards to find the comps that actually win.
- Next: add item features, then XGBoost + SHAP on placement.